# VQE for a spin model using `qse` Operators

This notebook builds a spin model using the `qse` `Operator` / `Operators` classes,
converts it to a Qiskit `SparsePauliOp` via `to_qiskit()` method, and runs VQE to find the ground state energy.

In [7]:
import qse
from qse.operator.operator import Operator
from qse.operator.operators import Operators

## 1. Build the spin model.

We build a 1D **transverse-field Ising model** (TFIM) on $N=10$ qubits:

$$H = -J \sum_{i=0}^{N-2} Z_i Z_{i+1} - h \sum_{i=0}^{N-1} X_i$$

In [27]:
N = 10
J = 1.0
h = 0.5
op_list = []

# ZZ interaction terms between neighbouring qubits
for i in range(N - 1):
    op_list.append(Operator(["Z", "Z"], [i, i + 1], N, coef=-J))

# Transverse field X terms on each qubit
for i in range(N):
    op_list.append(Operator("X", i, N, coef=-h))

H = Operators(op_list)
print(H)

Number of qubits: 10
Number of terms: 19

-1.00 Z0 Z1
-1.00 Z1 Z2
-1.00 Z2 Z3
-1.00 Z3 Z4
-1.00 Z4 Z5
-1.00 Z5 Z6
-1.00 Z6 Z7
-1.00 Z7 Z8
-1.00 Z8 Z9
-0.50 X0
-0.50 X1
-0.50 X2
-0.50 X3
-0.50 X4
-0.50 X5
-0.50 X6
-0.50 X7
-0.50 X8
-0.50 X9


## 2. Convert to Qiskit `SparsePauliOp`

Once we have the Hamiltonian $H$, we can use the `to_qiskit()` method to transform it to a Qiskit Sparse Pauli Operator. 

In [11]:
pauli_op = H.to_qiskit()
print(pauli_op)

SparsePauliOp(['ZZIIIIIIII', 'IZZIIIIIII', 'IIZZIIIIII', 'IIIZZIIIII', 'IIIIZZIIII', 'IIIIIZZIII', 'IIIIIIZZII', 'IIIIIIIZZI', 'IIIIIIIIZZ', 'XIIIIIIIII', 'IXIIIIIIII', 'IIXIIIIIII', 'IIIXIIIIII', 'IIIIXIIIII', 'IIIIIXIIII', 'IIIIIIXIII', 'IIIIIIIXII', 'IIIIIIIIXI', 'IIIIIIIIIX'],
              coeffs=[-1. +0.j, -1. +0.j, -1. +0.j, -1. +0.j, -1. +0.j, -1. +0.j, -1. +0.j,
 -1. +0.j, -1. +0.j, -0.5+0.j, -0.5+0.j, -0.5+0.j, -0.5+0.j, -0.5+0.j,
 -0.5+0.j, -0.5+0.j, -0.5+0.j, -0.5+0.j, -0.5+0.j])


## 3. Set up and run VQE.

We set up and run the VQE in qiskit. 

In [25]:
from qiskit.primitives import StatevectorEstimator
from qiskit.circuit.library import efficient_su2

import numpy as np
from scipy.optimize import minimize

In [ ]:
ansatz = efficient_su2(num_qubits=N, reps=2, entanglement="linear")
estimator = StatevectorEstimator()
energies_list = []


def cost_function(params):
    result = estimator.run([(ansatz, pauli_op, params)]).result()
    energy = result[0].data.evs
    energies_list.append(float(energy))
    return float(energy)


# Random initial parameters
rng = np.random.default_rng(42)
x0 = rng.uniform(-np.pi, np.pi, ansatz.num_parameters)

result = minimize(
    cost_function, x0, method="COBYLA", options={"maxiter": 2000, "rhobeg": 0.5}
)

VQE ground state energy : -9.745759


In [30]:
ground_state = np.linalg.eigh(pauli_op.to_matrix())[0][0]
print(f"VQE ground state energy : {result.fun:.6f}")
print(f"Exact ground state energy: {ground_state}")

VQE ground state energy : -9.745759
Exact ground state energy: -9.76550395792718
